# EDA : Target and Temporal Analysis

This notebook explores climate risk class distributions, sea surface temperature anomalies (SSTA), seasonality, autocorrelation, and potential lead-time relationships using the TAO dataset.


In [1]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

train = pd.read_csv("../data/processed/tao-all2-train.csv")
val = pd.read_csv("../data/processed/tao-all2-val.csv")
test = pd.read_csv("../data/processed/tao-all2-test.csv")

# Run from the repo root whether the kernel starts in notebooks/ (VS Code default) or the root
if Path.cwd().name == "notebooks":
    os.chdir("..")
print("Working directory:", Path.cwd())

Working directory: c:\Users\sdhar\OneDrive\Desktop\School Documents\BTT\AWS-1A-climate-risk-modeling-for-supply-chain-resiliency


In [2]:
print("Training:", train.shape)
print("Validation:", val.shape)
print("Testing:", test.shape)

print("\n Columns:")
print(train.columns.tolist())

Training: (67892, 30)
Validation: (62620, 30)
Testing: (30561, 30)

 Columns:
['obs', 'year', 'month', 'day', 'latitude', 'longitude', 'zonal_wind', 'meridional_wind', 'humidity', 'air_temp', 'ss_temp', 'standardized_date', 'buoy_segment_id', 'nominal_lat', 'nominal_lon', 'site_id', 'monthly_clim', 'sst_anomaly', 'risk_class', 'temp_lag_30', 'temp_mean_30d', 'temp_std_30d', 'temp_lag_90', 'temp_mean_90d', 'temp_std_90d', 'zonal_wind_scaled', 'meridional_wind_scaled', 'humidity_scaled', 'air_temp_scaled', 'ss_temp_scaled']


In [3]:
print(train["risk_class"].value_counts(dropna=False))

risk_class
Neutral          39209
Moderate Warm    12521
Moderate Cool     9876
Extreme Cool      3601
Extreme Warm      2685
Name: count, dtype: int64


In [4]:
# Comparing risk class distributions across data splits

for name, df in [
    ("Training", train),
    ("Validation", val),
    ("Testing", test)
]:
    print(f"\n{name}:")
    print(
        (df["risk_class"].value_counts(normalize=True, dropna=False) * 100).round(2)
    )


Training:
risk_class
Neutral          57.75
Moderate Warm    18.44
Moderate Cool    14.55
Extreme Cool      5.30
Extreme Warm      3.95
Name: proportion, dtype: float64

Validation:
risk_class
Neutral          40.38
Moderate Cool    21.50
Moderate Warm    15.53
NaN              13.68
Extreme Cool      7.61
Extreme Warm      1.30
Name: proportion, dtype: float64

Testing:
risk_class
Neutral          34.14
Extreme Warm     17.46
Moderate Warm    16.06
NaN              15.69
Moderate Cool    14.49
Extreme Cool      2.16
Name: proportion, dtype: float64


# Missing Target Labels

### Findings
The validation and testing sets contain 8,568 (13.68%) and 4,795 (15.69%) missing risk labels, respectively. These correspond to missing SSTA values caused by unavailable monthly climatology baselines, rather than missing SST observations.

This is consistent with the preprocessing approach, which calculates the historical baselines using training data only to prevent data leakage. However, the missing labels reduce the number of observations available for evaluation.

In [5]:
for name, df in [
    ("Training", train),
    ("Validation", val),
    ("Testing", test)
]:
    print(f"\n {name}:")
    print("Missing risk classes:", df["risk_class"].isna().sum())
    print("Missing SSTA:", df["sst_anomaly"].isna().sum())


 Training:
Missing risk classes: 0
Missing SSTA: 0

 Validation:
Missing risk classes: 8567
Missing SSTA: 8567

 Testing:
Missing risk classes: 4795
Missing SSTA: 4795


In [6]:
for name, df in [
    ("Validation", val),
    ("Testing", test)
]:
    missing = df[df["risk_class"].isna()]

    print(f"\n{name}:")
    print("Missing monthly climatology:",
          missing["monthly_clim"].isna().sum())

    print("Missing SST:", missing["ss_temp"].isna().sum())


Validation:
Missing monthly climatology: 8567
Missing SST: 0

Testing:
Missing monthly climatology: 4795
Missing SST: 0


# Class Distribution Across Data Splits


In [7]:
class_order = [
    "Extreme Cool",
    "Moderate Cool",
    "Neutral",
    "Moderate Warm",
    "Extreme Warm"
]

class_distribution = pd.DataFrame({
    "Training": train["risk_class"].value_counts(normalize=True) * 100,
    "Validation": val["risk_class"].value_counts(normalize=True) * 100,
    "Testing": test["risk_class"].value_counts(normalize=True) * 100
}).reindex(class_order)

display(class_distribution.round(2))

,Training,Validation,Testing
risk_class,,,
Extreme Cool,5.30,8.82,2.56
Moderate Cool,14.55,24.91,17.18
Neutral,57.75,46.78,40.50
Moderate Warm,18.44,17.99,19.05
Extreme Warm,3.95,1.50,20.71
